# 01 - Thu thập dữ liệu đánh giá Laptop trên Shopee
Notebook này tái sử dụng pattern từ bài TH5/TH6: request, phân trang, xử lý lỗi, chống trùng, ghép DataFrame và lưu CSV UTF-8.

In [ ]:
%pip install requests pandas

In [1]:
import re
import time
from datetime import datetime

import pandas as pd
import requests

In [ ]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://shopee.vn/"
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS)

def extract_ids_from_url(product_url: str):
    match = re.search(r"-i\.(\d+)\.(\d+)", product_url)
    if not match:
        raise ValueError(f"URL không đúng định dạng Shopee: {product_url}")
    return int(match.group(1)), int(match.group(2))

def fetch_ratings_page(shop_id: int, item_id: int, offset: int = 0, limit: int = 50):
    endpoint = "https://shopee.vn/api/v2/item/get_ratings"
    params = {
        "filter": 0,
        "flag": 1,
        "itemid": item_id,
        "shopid": shop_id,
        "limit": limit,
        "offset": offset,
        "type": 0
    }

    response = SESSION.get(endpoint, params=params, timeout=20)
    response.raise_for_status()
    payload = response.json()

    if payload.get("error") not in (0, None):
        raise RuntimeError(f"Shopee API trả lỗi: {payload.get('error')}")

    return payload.get("data", {}).get("ratings", [])

def parse_rating_row(rating_row: dict, shop_id: int, item_id: int):
    ctime = rating_row.get("ctime")
    created_at = datetime.fromtimestamp(ctime).isoformat() if ctime else None

    user_id = rating_row.get("userid")
    cmt_id = rating_row.get("cmtid")

    return {
        "review_id": f"{user_id}_{cmt_id}",
        "shop_id": shop_id,
        "item_id": item_id,
        "user_id": user_id,
        "rating_star": rating_row.get("rating_star"),
        "comment": (rating_row.get("comment") or "").strip(),
        "created_at": created_at,
        "like_count": rating_row.get("like_count", 0),
        "product_items": ", ".join([i.get("name", "") for i in rating_row.get("product_items", [])]),
        "source": "Shopee"
    }

def crawl_product_reviews(product_url: str, max_reviews: int = 500, page_size: int = 50, delay_seconds: float = 1.2):
    shop_id, item_id = extract_ids_from_url(product_url)

    reviews = []
    seen_review_ids = set()
    offset = 0
    empty_or_error_pages = 0

    while len(reviews) < max_reviews:
        try:
            page_reviews = fetch_ratings_page(shop_id=shop_id, item_id=item_id, offset=offset, limit=page_size)

            if not page_reviews:
                empty_or_error_pages += 1
                if empty_or_error_pages >= 3:
                    print(f"Dừng {product_url}: không còn dữ liệu sau {empty_or_error_pages} trang liên tiếp.")
                    break
                offset += page_size
                time.sleep(delay_seconds)
                continue

            empty_or_error_pages = 0

            for row in page_reviews:
                if len(reviews) >= max_reviews:
                    break

                parsed = parse_rating_row(row, shop_id=shop_id, item_id=item_id)
                if not parsed["comment"]:
                    continue
                if parsed["review_id"] in seen_review_ids:
                    continue

                seen_review_ids.add(parsed["review_id"])
                reviews.append(parsed)

            print(f"{product_url} -> Đã lấy {len(reviews)} review")
            offset += page_size
            time.sleep(delay_seconds)

        except Exception as exc:
            empty_or_error_pages += 1
            print(f"Lỗi tại offset={offset} ({product_url}): {exc}")
            if empty_or_error_pages >= 3:
                print("Dừng sản phẩm do lỗi liên tiếp.")
                break
            offset += page_size
            time.sleep(delay_seconds)

    return pd.DataFrame(reviews)

In [ ]:
product_urls = [
    # Thay bằng URL laptop thực tế bạn muốn crawl
    # Ví dụ: "https://shopee.vn/ten-san-pham-i.123456.987654321"
]

if not product_urls:
    raise ValueError("Bạn cần điền danh sách product_urls trước khi chạy crawl.")

In [ ]:
all_frames = []
for url in product_urls:
    df_item = crawl_product_reviews(url, max_reviews=400, page_size=50, delay_seconds=1.2)
    if not df_item.empty:
        all_frames.append(df_item)

if not all_frames:
    raise RuntimeError("Không thu được dữ liệu. Kiểm tra lại URL hoặc thử chạy lại với mạng khác.")

df_raw = pd.concat(all_frames, ignore_index=True)
df_raw = df_raw.drop_duplicates(subset=["review_id"]).reset_index(drop=True)

display(df_raw.head())
display(df_raw.tail())
print(f"Tổng số review thu được: {len(df_raw)}")

In [ ]:
output_path = "data/shopee_laptop_raw.csv"
df_raw.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Đã lưu dữ liệu thô vào: {output_path}")

## Ghi chú
Nếu Shopee chặn truy cập API ở một số thời điểm, bạn có thể: tăng `delay_seconds`, đổi mạng, hoặc giảm số sản phẩm trong một lần chạy để ổn định hơn.